**CI twin of `ch03-layers-networks.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
import numpy as np
import math

def neuron(x, w, b):                      # Chapter 1's function, verbatim
    z = sum(xi * wi for xi, wi in zip(x, w)) + b
    return 1 / (1 + math.exp(-z))

x = [2.0, 1.0]
weights = [[0.5, -1.0], [1.0, 1.0], [-0.5, 0.25]]
biases = [0.0, -1.0, 0.5]

looped = [neuron(x, w, b) for w, b in zip(weights, biases)]

W = np.array(weights)                     # one ROW per neuron
b = np.array(biases)
vectorized = 1 / (1 + np.exp(-(W @ np.array(x) + b)))

print("one at a time:", [round(v, 6) for v in looped])
print("all at once:  ", np.round(vectorized, 6).tolist())

In [ ]:
def count_params(sizes):
    """Total weights + biases for a fully-connected net, e.g. [64, 16, 10]."""
    return sum(sizes[i] * sizes[i + 1] + sizes[i + 1]
               for i in range(len(sizes) - 1))

print(f"Chapter 1's lone neuron (64 -> 1):   {count_params([64, 1]):>6}")
print(f"this chapter's net (64 -> 16 -> 10): {count_params([64, 16, 10]):>6}")

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(
    digits.data, digits.target, test_size=0.25,
    random_state=42, stratify=digits.target)

net = MLPClassifier(hidden_layer_sizes=(16,), activation="relu",
                    random_state=0, max_iter=2000).fit(Xtr, ytr)
print(f"held-out accuracy (10 classes): "
      f"{accuracy_score(yte, net.predict(Xte)):.3f}")
print(f"weight matrices: {[c.shape for c in net.coefs_]}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 8, figsize=(8.5, 2.4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(net.coefs_[0][:, i].reshape(8, 8), cmap="coolwarm")
    ax.set_title(f"clerk {i}", fontsize=6)
    ax.axis("off")
plt.show()

In [ ]:
def hidden_summary(x):
    return np.maximum(0, x @ net.coefs_[0] + net.intercepts_[0])

eights = np.where(yte == 8)[0][:2]
a_one = np.where(yte == 1)[0][0]
h8a = hidden_summary(Xte[eights[0]])
h8b = hidden_summary(Xte[eights[1]])
h1 = hidden_summary(Xte[a_one])

print("an 8's summary:", np.round(h8a, 1))

def cosine(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

print(f"\nsimilarity of two different 8s:   {cosine(h8a, h8b):.3f}")
print(f"similarity of an 8 and a 1:       {cosine(h8a, h1):.3f}")

In [ ]:
import numpy as np
import math

def neuron(x, w, b):
    z = sum(xi * wi for xi, wi in zip(x, w)) + b
    return 1 / (1 + math.exp(-z))

x = [2.0, 1.0]
weights = [[0.5, -1.0], [1.0, 1.0], [-0.5, 0.25]]
biases = [0.0, -1.0, 0.5]
looped = [round(neuron(x, w, b), 6) for w, b in zip(weights, biases)]

W = np.array(weights)
b = np.array(biases)
z_all = W @ np.array(x) + b
vectorized = 1 / (1 + np.exp(-z_all))

run_tests([
    ("the matrix runs the whole panel",
     [round(float(v), 6) for v in vectorized], looped),
    ("shape grammar: (n_out,)", vectorized.shape, (3,)),
])

In [ ]:
import numpy as np

def layer(x, W, b, act=None):
    z = W @ x + b
    return z if act is None else act(z)

def count_params(sizes):
    return sum(sizes[i] * sizes[i + 1] + sizes[i + 1]
               for i in range(len(sizes) - 1))

W_fix = np.array([[1.0, 0.0], [0.0, 2.0]])
b_fix = np.array([0.0, -1.0])
x_fix = np.array([3.0, 0.0])

run_tests([
    ("linear layer", layer(x_fix, W_fix, b_fix).tolist(), [3.0, -1.0]),
    ("relu layer kills the negative",
     layer(x_fix, W_fix, b_fix, act=lambda z: np.maximum(0, z)).tolist(),
     [3.0, 0.0]),
    ("shape transformer 2 -> 2", layer(x_fix, W_fix, b_fix).shape, (2,)),
    ("the chapter's network", count_params([64, 16, 10]), 1210),
    ("a tiny net", count_params([2, 3, 1]), 13),
    ("no hidden layer at all", count_params([10, 10]), 110),
])